# Lab 3: Environment Check

Name: *(your name)*  
Date: *(today)*

## 1. Versions

In [ ]:
import sys, platform
from importlib.metadata import version, PackageNotFoundError
import rasterio

print("Python      ", sys.version.split()[0], "on", platform.system())
print("Environment ", sys.prefix)
print("GDAL        ", rasterio.__gdal_version__)
print()

packages = [
    "numpy", "scipy", "pandas", "bottleneck", "netcdf4", "openpyxl",
    "rasterio", "rioxarray", "xarray", "xarray-spatial", "xvec", "geopandas",
    "shapely", "pyproj", "exactextract", "odc-algo",
    "pystac-client", "planetary-computer", "odc-stac", "earthengine-api",
    "xee", "botocore", "scikit-learn", "omnicloudmask", "geotessera",
    "dask", "distributed", "coiled", "duckdb", "matplotlib", "lonboard",
    "graphviz", "jupyterlab", "aef-loader", "xrscipy",
]
missing = []
for name in packages:
    try:
        print(f"{name:<20}", version(name))
    except PackageNotFoundError:
        print(f"{name:<20}", "NOT INSTALLED")
        missing.append(name)
print()
print("All packages found." if not missing else f"Missing: {missing}")

## 2. Get a small raster

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

url = "https://raw.githubusercontent.com/rasterio/rasterio/main/tests/data/RGB.byte.tif"
path = Path("RGB.byte.tif")
if not path.exists():
    urlretrieve(url, path)
print(path, round(path.stat().st_size / 1e6, 2), "MB on disk")

## 3. Open it and read its metadata

In [ ]:
with rasterio.open(path) as src:
    print("Rows x columns :", src.height, "x", src.width)
    print("Bands          :", src.count)
    print("Data type      :", src.dtypes[0])
    print("CRS            :", src.crs)
    print("Pixel size     :", src.res)
    print("Bounds         :", src.bounds)
    img = src.read()          # array shaped (bands, rows, columns)

print("Array shape    :", img.shape)

## 4. Check the data-size formula from lecture

In [ ]:
bands, rows, cols = img.shape
bytes_per_pixel = img.dtype.itemsize
by_formula = rows * cols * bands * bytes_per_pixel

print("rows x cols x bands x bytes =", f"{by_formula:,}", "bytes")
print("NumPy reports (img.nbytes)  =", f"{img.nbytes:,}", "bytes")
print("Match:", by_formula == img.nbytes)

## 5. Display it

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rgb = np.moveaxis(img, 0, -1)   # (bands, rows, cols) -> (rows, cols, bands)

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(rgb)
ax[0].set_title("Bands 1-2-3 as R-G-B")
im = ax[1].imshow(img[0], cmap="viridis")
ax[1].set_title("Band 1 alone, viridis color ramp")
fig.colorbar(im, ax=ax[1], shrink=0.8, label="DN")
for a in ax:
    a.set_xlabel("column"); a.set_ylabel("row")
plt.tight_layout()
plt.show()

## What went wrong during installation, and how did you fix it?

*(Write your one- or two-sentence answer here. Write "nothing" if it all worked.)*